### Packages

In [ ]:
import scanpy as sc
import anndata as ad
import torch
import collections
from torch import nn
from collections.abc import Iterable
from typing import Callable, Literal, Optional
from scipy.sparse import csr_matrix
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.distributions import Normal, Poisson
from torch.distributions import kl_divergence as kl
from torch.optim import Adam
from torch import logsumexp
import scvi
from sklearn.preprocessing import LabelEncoder
import torch.nn.functional as F
import pandas as pd

### Model

In [ ]:
## Encoder
class PeakANVI(nn.Module):
    def __init__(
        self, 
        layer_sizes: list,
        classifier_layer_sizes: list,
        n_labels: int,
        n_input: int,
        embedding_dim: int,
        n_samples = int,
        n_study = int,
        latent_size: int = 25,
        use_batch_norm: bool = True,
        use_layer_norm: bool = True,
        classifier_use_batch_norm: bool = True,
        classifier_use_layer_norm: bool = False,
        dropout_rate: float = 0.1,
        classifier_dropout_rate: float = 0.1,
        activation_function: nn.Module = nn.LeakyReLU(),
        classifier_activation_function: nn.Module = nn.ReLU(),
        use_data_bias: bool = True,
        use_cat_bias: bool = False,
        linear_classifier = True,
    ):
        super().__init__()
        self.use_batch_norm = use_batch_norm
        self.use_layer_norm = use_layer_norm
        self.classifier_use_batch_norm = classifier_use_batch_norm
        self.classifier_use_layer_norm = classifier_use_layer_norm
        self.dropout_rate = dropout_rate
        self.classifier_dropout_rate = classifier_dropout_rate
        self.linear_classifier = linear_classifier
        
        ## Encoder
        # First layer
        self.encoder_first = nn.Linear(n_input, layer_sizes[0], bias=use_data_bias)
        self.encoder_celltype_layer = nn.Linear(n_labels, layer_sizes[0], bias=use_cat_bias)
        self.encoder_batch_layer = nn.Linear(n_study, layer_sizes[0], bias=use_cat_bias)
        self.encoder_embed_layer = nn.Linear(embedding_dim, layer_sizes[0], bias=use_cat_bias)
        if use_batch_norm:
            self.encoder_first_bn = nn.BatchNorm1d(layer_sizes[0], affine=True)
        if use_layer_norm:
            self.encoder_first_ln = nn.LayerNorm(layer_sizes[0], elementwise_affine=False)
        self.encoder_first_activation = activation_function
        if dropout_rate > 0:
            self.encoder_first_dropout = nn.Dropout(p=dropout_rate)
        
        # Additional layers
        self.n_additional_layers = len(layer_sizes) - 1
        additional_layer_sizes = layer_sizes + [layer_sizes[len(layer_sizes)-1]]
        if self.n_additional_layers > 0:
            self.encoder_add_linear = torch.nn.ModuleList([nn.Linear(additional_layer_sizes[i],additional_layer_sizes[i+1],bias = use_data_bias) for i in range(self.n_additional_layers)])
            if use_batch_norm:
                self.encoder_add_bn = torch.nn.ModuleList([nn.BatchNorm1d(additional_layer_sizes[i+1], affine=True) for i in range(self.n_additional_layers)])
            if use_layer_norm:
                self.encoder_add_ln = torch.nn.ModuleList([nn.LayerNorm(additional_layer_sizes[i+1], elementwise_affine=False) for i in range(self.n_additional_layers)])
            self.encoder_add_activations = torch.nn.ModuleList([activation_function for i in range(self.n_additional_layers)])
            if dropout_rate > 0:
                self.encoder_add_dropouts = torch.nn.ModuleList([nn.Dropout(p=dropout_rate) for i in range(self.n_additional_layers)])
        
        # Mean and variance encoder
        self.mean_encoder = nn.Linear(layer_sizes[-1], latent_size)
        self.var_encoder = nn.Linear(layer_sizes[-1], latent_size)
        
        # Sample embedding
        self.embedding = nn.Embedding(n_samples, embedding_dim, max_norm=1)

        ## Decoder
        layer_sizes.reverse()
        
        # First layer
        self.decoder_first = nn.Linear(latent_size, layer_sizes[0], bias=use_data_bias)
        self.decoder_celltype_layer = nn.Linear(n_labels, layer_sizes[0], bias=use_cat_bias)
        self.decoder_batch_layer = nn.Linear(n_study, layer_sizes[0], bias=use_cat_bias)
        self.decoder_embed_layer = nn.Linear(embedding_dim, layer_sizes[0], bias=use_cat_bias)
        if use_batch_norm:
            self.decoder_first_bn = nn.BatchNorm1d(layer_sizes[0], affine=True)
        if use_layer_norm:
            self.decoder_first_ln = nn.LayerNorm(layer_sizes[0], elementwise_affine=False)
        self.decoder_first_activation = activation_function
        if dropout_rate > 0:
            self.decoder_first_dropout = nn.Dropout(p=dropout_rate)
        
        # Additional layers
        self.n_additional_layers = len(layer_sizes) - 1
        additional_layer_sizes = layer_sizes + [layer_sizes[len(layer_sizes)-1]]
        if self.n_additional_layers > 0:
            self.decoder_add_linear = torch.nn.ModuleList([nn.Linear(additional_layer_sizes[i],additional_layer_sizes[i+1],bias = use_data_bias) for i in range(self.n_additional_layers)])
            if use_batch_norm:
                self.decoder_add_bn = torch.nn.ModuleList([nn.BatchNorm1d(additional_layer_sizes[i+1], affine=True) for i in range(self.n_additional_layers)])
            if use_layer_norm:
                self.decoder_add_ln = torch.nn.ModuleList([nn.LayerNorm(additional_layer_sizes[i+1], elementwise_affine=False) for i in range(self.n_additional_layers)])
            self.decoder_add_activations = torch.nn.ModuleList([activation_function for i in range(self.n_additional_layers)])
            if dropout_rate > 0:
                self.decoder_add_dropouts = torch.nn.ModuleList([nn.Dropout(p=dropout_rate) for i in range(self.n_additional_layers)])

        # Reconstruction layers
        self.decoder_reconstruct = nn.Sequential(
            nn.Linear(layer_sizes[-1], n_input, bias=use_data_bias), 
            nn.Softmax(dim=-1)
        )

        ## Classifier
        if linear_classifier:
            self.classifier_out = nn.Linear(latent_size, n_labels, bias = True)
        else:            
            self.n_classifier_layers = len(classifier_layer_sizes)
            classifier_layer_sizes = [latent_size] + classifier_layer_sizes
            self.classifier_linear = torch.nn.ModuleList([nn.Linear(classifier_layer_sizes[i],classifier_layer_sizes[i+1],bias = True) for i in range(self.n_classifier_layers)])
            if classifier_use_batch_norm:
                self.classifier_bn = torch.nn.ModuleList([nn.BatchNorm1d(classifier_layer_sizes[i+1], affine=True) for i in range(self.n_classifier_layers)])
            if classifier_use_layer_norm:
                self.classifier_ln = torch.nn.ModuleList([nn.LayerNorm(classifier_layer_sizes[i+1], elementwise_affine=False) for i in range(self.n_classifier_layers)])
            if classifier_dropout_rate > 0:
                self.classifier_dropout = torch.nn.ModuleList([nn.Dropout(p=classifier_dropout_rate) for i in range(self.n_classifier_layers)])
            self.classifier_activation = torch.nn.ModuleList([classifier_activation_function for i in range(self.n_classifier_layers)])    
            self.classifier_out = nn.Linear(classifier_layer_sizes[-1], n_labels, bias = True)

    def forward(self, x: torch.Tensor, batch: torch.Tensor, ct: torch.Tensor, e: torch.Tensor, encoder_inject_embed = False, encoder_inject_celltypes = True, encoder_inject_batches = False, decoder_inject_celltypes = True, decoder_inject_embed = True, decoder_inject_batches = True):
        ## Setup
        x_ = x
        x = torch.log(1 + x)
        
        ## Embedding
        e = self.embedding(e)
        
        ## Encoder
        # First layer
        x = self.encoder_first(x)
        if encoder_inject_celltypes:
            x = x + self.encoder_celltype_layer(ct)
        if encoder_inject_batches:
            x = x + self.encoder_batch_layer(batch)
        if encoder_inject_embed:
            x = x + self.encoder_embed_layer(e)
        if self.use_batch_norm:
            x = self.encoder_first_bn(x)
        if self.use_layer_norm:
            x = self.encoder_first_ln(x)
        x = self.encoder_first_activation(x)
        if self.dropout_rate > 0:
            x = self.encoder_first_dropout(x)

        # Additional layers
        if self.n_additional_layers > 0:
            for i in range(self.n_additional_layers):
                x = self.encoder_add_linear[i](x)
                if self.use_batch_norm:
                    x = self.encoder_add_bn[i](x)
                if self.use_layer_norm:
                    x = self.encoder_add_ln[i](x)
                x = self.encoder_add_activations[i](x)
                if self.dropout_rate > 0:
                    x = self.encoder_add_dropouts[i](x)

        # Latent parameters
        mu = self.mean_encoder(x)
        log_var = self.var_encoder(x)

        # Sample from distribution with reparameterization trick
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        z = mu + eps * std

        # Decoder
        yhat = self.decoder_first(z)
        if decoder_inject_celltypes:
            yhat = yhat + self.decoder_celltype_layer(ct)
        if decoder_inject_batches:
            yhat = yhat + self.decoder_batch_layer(batch)
        if decoder_inject_embed:
            yhat = yhat + self.decoder_embed_layer(e)
        if self.use_batch_norm:
            yhat = self.decoder_first_bn(yhat)
        if self.use_layer_norm:
            yhat = self.encoder_first_ln(yhat)
        yhat = self.decoder_first_activation(yhat)
        if self.dropout_rate > 0:
            yhat = self.encoder_first_dropout(yhat)

        # Additional layers
        if self.n_additional_layers > 0:
            for i in range(self.n_additional_layers):
                yhat = self.decoder_add_linear[i](yhat)
                if self.use_batch_norm:
                    yhat = self.decoder_add_bn[i](yhat)
                if self.use_layer_norm:
                    yhat = self.decoder_add_ln[i](yhat)
                yhat = self.decoder_add_activations[i](yhat)
                if self.dropout_rate > 0:
                    yhat = self.decoder_add_dropouts[i](yhat)

        # Distribution parameters
        scale = self.decoder_reconstruct(yhat)
        rate = torch.exp(torch.log(x_.sum(1)).unsqueeze(1)) * scale
        px = Poisson(rate)
         
        ## Classifier
        cprob = mu
        if not self.linear_classifier:
            for i in range(self.n_classifier_layers):
                cprob = self.classifier_linear[i](cprob)
                if self.classifier_use_batch_norm:
                    cprob = self.classifier_bn[i](cprob)
                if self.classifier_use_layer_norm:
                    cprob = self.classifier_ln[i](cprob)
                cprob = self.classifier_activation[i](cprob)
                if self.classifier_dropout_rate > 0:
                    cprob = self.classifier_dropout[i](cprob)                   
        class_out = self.classifier_out(cprob)

        ## Outputs
        outputs = {"z": z, "mu": mu, "log_var": log_var, "px": px, "labels_classifier": class_out, "embedding": e}        
        return(outputs) 

### Data

In [ ]:
# Load the data and setup
from sklearn.preprocessing import OneHotEncoder
from tqdm.notebook import tqdm
metadata = pd.read_csv('../../data/public/scATACseq/Project/IC_metadata.csv', index_col=0)
adata = ad.read_h5ad('../../data/public/scATACseq/Project/full_adata_filtered_10.h5ad')
sample_index = pd.factorize(adata.obs.Sample)[0]
encoder = OneHotEncoder(sparse_output=False)
celltypes = encoder.fit_transform(np.array(adata.obs.labels).reshape(-1, 1))
encoder = OneHotEncoder(sparse_output=False)
batches = encoder.fit_transform(np.array(adata.obs.study).reshape(-1, 1))

In [ ]:
# Get dimensions
ncells, npeaks = adata.shape
indices = np.arange(ncells)
n_samples = len(np.unique(sample_index))
n_labels = celltypes.shape[1]
n_studies = batches.shape[1]
print("Data contains " + str(ncells) + " cells and " + str(npeaks) + " peaks distributed across " + str(n_samples) + " samples in " + str(n_studies) + " studies and " + str(n_labels) + " cell types.")

### Unsupervised training

In [ ]:
# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
latent_dim = 25
embedding_dim = 20
layer_sizes = [256,128]
classifier_layer_sizes = [32]
model = PeakANVI(layer_sizes = layer_sizes, n_study=n_studies, n_labels = n_labels, classifier_layer_sizes = classifier_layer_sizes, n_samples = n_samples, n_input = npeaks, embedding_dim = embedding_dim, latent_size = latent_dim, use_layer_norm=False, use_batch_norm=True).to(device)

In [ ]:
print(model)

In [ ]:
import os
os.makedirs('../../data/project/PeakANVI', exist_ok=True)  # model checkpoints and embeddings

batch_size = 256
classifaction = False
kl_weight = 1
class_weight = 1
cel = nn.CrossEntropyLoss()
epochs = 100
pretrain = -1
optimizer = Adam(model.parameters(), lr=0.001)
model.train()

# Freeze layers used for classification
for name, param in model.named_parameters():
    if ('classifier' in name) or ('celltype' in name):
        param.requires_grad = False

In [ ]:
for epoch in tqdm(range(epochs), desc = 'Training'):
    # Loss trackers
    np.random.shuffle(indices) 
    classifier_loss = 0
    kl_loss = 0
    recon_loss = 0
    
    # Trigger classification loss
    if epoch >= epochs - pretrain:
        classifaction = True
        for name, param in model.named_parameters():
            if ('classifier' in name) or ('celltype' in name):
                param.requires_grad = True
  
    for i in range(0, ncells, batch_size):
        # Get training data for current mini-batch        
        batch = indices[i:i + batch_size]
        train_data = torch.Tensor(adata.X[batch,:].toarray()).to(device)
        train_samples = torch.Tensor(sample_index[batch]).type(torch.int).to(device)
        train_celltypes = torch.Tensor(celltypes[batch,]).to(device)
        train_batch = torch.Tensor(batches[batch,]).to(device)
    
        # Calculate outputs
        optimizer.zero_grad()
        if classifaction:
            output = model(x = train_data, batch = train_batch, ct = train_celltypes, e = train_samples, encoder_inject_embed = True, encoder_inject_celltypes = False, encoder_inject_batches = False, decoder_inject_celltypes = False, decoder_inject_embed = True, decoder_inject_batches = False)
        else:
            output = model(x = train_data, batch = train_batch, ct = train_celltypes, e = train_samples, encoder_inject_embed = True, encoder_inject_celltypes = False, encoder_inject_batches = False, decoder_inject_celltypes = False, decoder_inject_embed = True, decoder_inject_batches = False)
    
        # Calculate ELBO loss
        kl_divergence_z = (-0.5 * torch.sum(1 + output["log_var"] - output["mu"].pow(2) - output["log_var"].exp())) / batch_size
        reconst_loss = torch.sum(-1 * output['px'].log_prob(train_data).sum(dim=-1)) / batch_size
        loss = reconst_loss + (kl_divergence_z * kl_weight)
        recon_loss += reconst_loss.item()
        kl_loss += kl_divergence_z.item()

        # Calculate classification loss
        if classifaction:
            labels_class_loss = cel(output['labels_classifier'],train_celltypes)
            loss = loss + (labels_class_loss * class_weight)
            classifier_loss += labels_class_loss.item()

        # Update the model parameters
        loss.backward()
        optimizer.step()

    # Print
    print("\tEpoch", epoch + 1, "complete!", "\tKL Loss: ", kl_loss, "\tReconstruction Loss: ", recon_loss, "\tClassifier Loss: ", classifier_loss)

    # Save model state
    if (epoch + 1) % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss,
        }, '../../data/project/PeakANVI/IC_10_PeakANVI_linear_epoch_nobatch_' + str((epoch+1)))

In [ ]:
# Get Z-space
model.eval()
indices = np.arange(ncells)
iter = 0 
with torch.no_grad():
    for i in tqdm(range(0, ncells, batch_size)):
            # Get training data for current mini-batch        
            batch = indices[i:i + batch_size]
            train_data = torch.Tensor(adata.X[batch,:].toarray()).to(device)
            train_samples = torch.Tensor(sample_index[batch]).type(torch.int).to(device)
            train_celltypes = torch.Tensor(celltypes[batch,]).to(device)
            train_batch = torch.Tensor(batches[batch,]).to(device)
    
            # Get output
            if classifaction:
                output = model(x = train_data, batch = train_batch, ct = train_celltypes, e = train_samples, encoder_inject_embed = True, encoder_inject_celltypes = False, encoder_inject_batches = False, decoder_inject_celltypes = False, decoder_inject_embed = True, decoder_inject_batches = False)
            else:
                output = model(x = train_data, batch = train_batch, ct = train_celltypes, e = train_samples, encoder_inject_embed = True, encoder_inject_celltypes = False, encoder_inject_batches = False, decoder_inject_celltypes = False, decoder_inject_embed = True, decoder_inject_batches = False)
       
            # Combine
            if iter == 0:
                Z = output['z'].cpu().numpy()
                E = output['embedding'].cpu().numpy()
                iter += 1
            else:
                Z = np.concatenate((Z, output['z'].cpu().numpy()))
                E = np.concatenate((E, output['embedding'].cpu().numpy()))

E = pd.merge(pd.DataFrame(E, index = metadata.index), metadata[['Sample','study','disease','gender','age','bmi','hba1c','ethnicity']], left_index = True, right_index = True)
E = E.drop_duplicates(subset=['Sample'])

In [ ]:
np.savez('../../data/project/PeakANVI/unsupervised.npz', arr1=Z)

### Supervised training

In [ ]:
# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model
latent_dim = 25
embedding_dim = 20
layer_sizes = [256,128]
classifier_layer_sizes = [32]
model = PeakANVI(layer_sizes = layer_sizes, n_study=n_studies, n_labels = n_labels, classifier_layer_sizes = classifier_layer_sizes, n_samples = n_samples, n_input = npeaks, embedding_dim = embedding_dim, latent_size = latent_dim, use_layer_norm=False, use_batch_norm=True).to(device)

In [ ]:
batch_size = 256
classifaction = True
kl_weight = 1
class_weight = 1
cel = nn.CrossEntropyLoss()
epochs = 5
pretrain = epochs + 1
optimizer = Adam(model.parameters(), lr=0.001)

checkpoint = torch.load('../../data/project/PeakANVI/IC_10_PeakANVI_linear_epoch_nobatch_100')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
model.train()

# Freeze layers used for classification
for name, param in model.named_parameters():
    if ('classifier' in name) or ('celltype' in name):
        param.requires_grad = False



In [ ]:
for epoch in tqdm(range(epochs), desc = 'Training'):
    # Loss trackers
    np.random.shuffle(indices) 
    classifier_loss = 0
    kl_loss = 0
    recon_loss = 0
    
    # Trigger classification loss
    if epoch >= epochs - pretrain:
        classifaction = True
        for name, param in model.named_parameters():
            if ('classifier' in name) or ('celltype' in name):
                param.requires_grad = True
  
    for i in range(0, ncells, batch_size):
        # Get training data for current mini-batch        
        batch = indices[i:i + batch_size]
        train_data = torch.Tensor(adata.X[batch,:].toarray()).to(device)
        train_samples = torch.Tensor(sample_index[batch]).type(torch.int).to(device)
        train_celltypes = torch.Tensor(celltypes[batch,]).to(device)
        train_batch = torch.Tensor(batches[batch,]).to(device)
    
        # Calculate outputs
        optimizer.zero_grad()
        if classifaction:
            output = model(x = train_data, batch = train_batch, ct = train_celltypes, e = train_samples, encoder_inject_embed = True, encoder_inject_celltypes = False, encoder_inject_batches = False, decoder_inject_celltypes = False, decoder_inject_embed = True, decoder_inject_batches = False)
        else:
            output = model(x = train_data, batch = train_batch, ct = train_celltypes, e = train_samples, encoder_inject_embed = True, encoder_inject_celltypes = False, encoder_inject_batches = False, decoder_inject_celltypes = False, decoder_inject_embed = True, decoder_inject_batches = False)
    
        # Calculate ELBO loss
        kl_divergence_z = (-0.5 * torch.sum(1 + output["log_var"] - output["mu"].pow(2) - output["log_var"].exp())) / batch_size
        reconst_loss = torch.sum(-1 * output['px'].log_prob(train_data).sum(dim=-1)) / batch_size
        loss = reconst_loss + (kl_divergence_z * kl_weight)
        recon_loss += reconst_loss.item()
        kl_loss += kl_divergence_z.item()

        # Calculate classification loss
        if classifaction:
            labels_class_loss = cel(output['labels_classifier'],train_celltypes)
            loss = loss + (labels_class_loss * class_weight)
            classifier_loss += labels_class_loss.item()

        # Update the model parameters
        loss.backward()
        optimizer.step()

    # Print
    print("\tEpoch", epoch + 1, "complete!", "\tKL Loss: ", kl_loss, "\tReconstruction Loss: ", recon_loss, "\tClassifier Loss: ", classifier_loss)

    # Save model state
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss,
        }, '../../data/project/PeakANVI/IC_10_PeakANVI_linear_epoch_nobatch_supervised_' + str((epoch+1)))

In [ ]:
# Get Z-space
model.eval()
indices = np.arange(ncells)
iter = 0 
with torch.no_grad():
    for i in tqdm(range(0, ncells, batch_size)):
            # Get training data for current mini-batch        
            batch = indices[i:i + batch_size]
            train_data = torch.Tensor(adata.X[batch,:].toarray()).to(device)
            train_samples = torch.Tensor(sample_index[batch]).type(torch.int).to(device)
            train_celltypes = torch.Tensor(celltypes[batch,]).to(device)
            train_batch = torch.Tensor(batches[batch,]).to(device)
    
            # Get output
            if classifaction:
                output = model(x = train_data, batch = train_batch, ct = train_celltypes, e = train_samples, encoder_inject_embed = True, encoder_inject_celltypes = False, encoder_inject_batches = False, decoder_inject_celltypes = False, decoder_inject_embed = True, decoder_inject_batches = False)
            else:
                output = model(x = train_data, batch = train_batch, ct = train_celltypes, e = train_samples, encoder_inject_embed = True, encoder_inject_celltypes = False, encoder_inject_batches = False, decoder_inject_celltypes = False, decoder_inject_embed = True, decoder_inject_batches = False)
       
            # Combine
            if iter == 0:
                Z = output['z'].cpu().numpy()
                E = output['embedding'].cpu().numpy()
                iter += 1
            else:
                Z = np.concatenate((Z, output['z'].cpu().numpy()))
                E = np.concatenate((E, output['embedding'].cpu().numpy()))

E = pd.merge(pd.DataFrame(E, index = metadata.index), metadata[['Sample','study','disease','gender','age','bmi','hba1c','ethnicity']], left_index = True, right_index = True)
E = E.drop_duplicates(subset=['Sample'])

In [ ]:
np.savez('../../data/project/PeakANVI/PeakANVI_final.npz', cells=Z, samples=E)
pd.DataFrame(Z).to_csv('../../data/public/scATACseq/JM/cells.csv', index=False)
pd.DataFrame(E.values).to_csv('../../data/public/scATACseq/JM/samples.csv', index=False)